In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -q pytorchvideo transformers evaluate

In [2]:
#Data collection

import os
import pandas as pd

root_path = "/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data"
folder_list = os.listdir(root_path)
label_list = [path for path in folder_list if not path.endswith((".csv"))]

train_df = pd.read_csv(os.path.join(root_path,"train.csv"))
test_df = pd.read_csv(os.path.join(root_path,"test.csv"))

selected_cols = ['label', 'video_name']
train_df = train_df[selected_cols]
test_df = test_df[selected_cols]
total_df = pd.concat([train_df, test_df])
total_df.reset_index(drop = True, inplace = True)
total_df['label'].value_counts()

label
normal           950
roadaccidents    150
robbery          150
stealing         100
burglary         100
explosion         50
fighting          50
vandalism         50
shoplifting       50
arrest            50
assault           50
shooting          50
arson             50
abuse             50
Name: count, dtype: int64

In [3]:
#Data split

from sklearn.model_selection import train_test_split

def correct_file_path(file_name: str, root_path: str):
    root_path = "/kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data"
    return os.path.join(root_path, file_name.replace('\\', '/'))

def preprocess_meta_df(df, root_path, label2id):
    df.rename(columns={"video_name": "video_path"}, inplace=True)
    df['video_path'] = df['video_path'].apply(lambda x: correct_file_path(x, root_path))
    df['label'] = df['label'].apply(lambda x: label2id[x])
    return df

selected_labels = ['normal', 'abuse', 'arson', 'burglary', 'explosion', 'roadaccidents', 'shooting']
total_df = total_df[total_df['label'].isin(selected_labels)]

sampling_meta_df, _ = train_test_split(total_df, test_size=0.1, stratify=total_df['label'], random_state=42)
train_meta_df, test_meta_df = train_test_split(sampling_meta_df, test_size=0.2, stratify=sampling_meta_df['label'], random_state=42)
test_meta_df, eval_meta_df = train_test_split(test_meta_df, test_size = 0.1, stratify=test_meta_df['label'], random_state=42)

label_list = list(set(train_meta_df['label']))
class_labels = sorted(label_list)
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}

print(f"Unique classes: {list(label2id.keys())}.")

train_meta_df = preprocess_meta_df(train_meta_df, root_path, label2id)
eval_meta_df = preprocess_meta_df(eval_meta_df, root_path, label2id)
test_meta_df = preprocess_meta_df(test_meta_df, root_path, label2id)

print("Splitted data:", len(train_meta_df), len(test_meta_df), len(eval_meta_df))

Unique classes: ['abuse', 'arson', 'burglary', 'explosion', 'normal', 'roadaccidents', 'shooting'].
Splitted data: 1008 226 26


In [4]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install pytorchvideo

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install utils

  Preparing metadata (setup.py) ... done
  Created wheel for utils: filename=utils-1.0.2-py2.py3-none-any.whl size=13906 sha256=556a2e15405ff2bab2435d89c1481fe195309bf0bfd18a1658c207bd6568e52e
  Stored in directory: /root/.cache/pip/wheels/b6/a1/81/1036477786ae0e17b522f6f5a838f9bc4288d1016fc5d0e1ec
Successfully built utils
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip  install torchvision

Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install transforms

Note: you may need to restart the kernel to use updated packages.


In [10]:
pip install functional

  Preparing metadata (setup.py) ... done
  Created wheel for functional: filename=functional-0.4-py3-none-any.whl size=5678 sha256=58a63e4d5e11ca3642436f70304762b937c37dbba8dcc4d6f436ee2fbb93367f
  Stored in directory: /root/.cache/pip/wheels/e5/0a/4d/de321ce5aba4b6814165111614469da891ffbdd7e5bae278f3
Successfully built functional
Note: you may need to restart the kernel to use updated packages.


In [11]:
#model seection
import torch
import sys


try:
    import torchvision.transforms.functional_tensor as T_f
except ImportError:
    import torchvision.transforms.functional as T_f
    sys.modules["torchvision.transforms.functional_tensor"] = T_f

from transformers import VivitImageProcessor, VivitForVideoClassification


class_labels = [
    "Abuse", "Arson", "Burglary", "Explosion", 
    "Normal", "RoadAccidents", "Shooting"
]

label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for i, label in enumerate(class_labels)}


model_checkpoint = "google/vivit-b-16x2-kinetics400"

image_processor = VivitImageProcessor.from_pretrained(model_checkpoint)
model = VivitForVideoClassification.from_pretrained(
    model_checkpoint, 
    label2id=label2id, 
    id2label=id2label, 
    ignore_mismatched_sizes=True
)

print(f" model successfully loaded with {len(id2label)} classes.")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

VivitForVideoClassification LOAD REPORT from: google/vivit-b-16x2-kinetics400
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400]) vs model:torch.Size([7])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([400, 768]) vs model:torch.Size([7, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


 model successfully loaded with 7 classes.


In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input/'):
    if any(f.endswith('.mp4') for f in files):
        print(f"Found videos in: {root}")
        break 

In [12]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torchvision.transforms import Compose, Lambda, Resize, RandomHorizontalFlip
from pytorchvideo.transforms import (
    ApplyTransformToKey,
    Normalize,
    RandomShortSideScale,
    UniformTemporalSubsample,
)

base_path = "/kaggle/input/"
dataset_root = None

for root, dirs, files in os.walk(base_path):
    if "roadaccidents" in dirs:
        dataset_root = root
        break

if dataset_root is None:
    raise FileNotFoundError(f"Deepesh, I couldn't find the 'roadaccidents' folder in {base_path}. Please check the Data tab on the right.")

def create_metadata_df(root_dir):
    data = []
   
    labels = sorted([f for f in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, f))])
    label2id = {label: i for i, label in enumerate(labels)}
    
    for label in labels:
        label_path = os.path.join(root_dir, label)
        for video_file in os.listdir(label_path):
            if video_file.lower().endswith(('.mp4', '.avi')):
                data.append({
                    'video_path': os.path.join(label_path, video_file),
                    'label': label2id[label]
                })
    return pd.DataFrame(data), label2id

full_df, label2id = create_metadata_df(dataset_root)

# 2. Data Splits
train_meta_df, temp_df = train_test_split(full_df, test_size=0.2, random_state=42)
eval_meta_df, test_meta_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# --- Preprocessing ---
mean = image_processor.image_mean
std = image_processor.image_std
resize_to = (model.config.image_size, model.config.image_size)
num_frames_to_sample = model.config.num_frames

train_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(num_frames_to_sample),
            Lambda(lambda x: x / 255.0),
            Normalize(mean, std),
            RandomShortSideScale(min_size=256, max_size=320),
            Resize(resize_to),
            RandomHorizontalFlip(p=0.5),
        ]),
    ),
])

# 3. Custom Dataset and Paths
class CustomVideoDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        return row['video_path'], row['label']

train_labeled_video_paths = [(v, {'label': l}) for v, l in CustomVideoDataset(train_meta_df)]
eval_labeled_video_paths = [(v, {'label': l}) for v, l in CustomVideoDataset(eval_meta_df)]
test_labeled_video_paths = [(v, {'label': l}) for v, l in CustomVideoDataset(test_meta_df)]

print(f"Deepesh, preprocessing is ready! Found {len(full_df)} videos.")
print(f"Training on path: {dataset_root}")

Deepesh, preprocessing is ready! Found 1900 videos.
Training on path: /kaggle/input/datasets/webadvisor/real-time-anomaly-detection-in-cctv-surveillance/data


In [13]:
#Data preprocessing

class CustomVideoDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        video_path = row['video_path']
        label = row['label']
        return video_path, label

mean = image_processor.image_mean
std = image_processor.image_std

if "shortest_edge" in image_processor.size:
    height = width = image_processor.size["shortest_edge"]
else:
    height = image_processor.size["height"]
    width = image_processor.size["width"]

resize_to = (model.config.image_size, model.config.image_size)
num_frames_to_sample = model.config.num_frames
sample_rate = 20
fps = 30
clip_duration = num_frames_to_sample * sample_rate / fps

train_transform = Compose(
    [
        ApplyTransformToKey(
            key="video",
            transform=Compose(
                [
                    UniformTemporalSubsample(num_frames_to_sample),
                    Lambda(lambda x: x / 255.0),
                    Normalize(mean, std),
                    RandomShortSideScale(min_size=256, max_size=320),
                    Resize(resize_to),
                    RandomHorizontalFlip(p=0.5),
                ]
            ),
        ),
    ]
)

val_transform = Compose(
    [
        ApplyTransformToKey(
            key="video",
            transform=Compose(
                [
                    UniformTemporalSubsample(num_frames_to_sample),
                    Lambda(lambda x: x / 255.0),
                    Normalize(mean, std),
                    Resize(resize_to),
                ]
            ),
        ),
    ]
)

train_custom_dataset = CustomVideoDataset(train_meta_df)
train_labeled_video_paths = [(video_path, {'label': label}) for video_path, label in train_custom_dataset]

eval_custom_dataset = CustomVideoDataset(eval_meta_df)
eval_labeled_video_paths = [(video_path, {'label': label}) for video_path, label in eval_custom_dataset]

test_custom_dataset = CustomVideoDataset(test_meta_df)
test_labeled_video_paths = [(video_path, {'label': label})  for video_path, label in test_custom_dataset]


In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input/'):
    if any(f.endswith('.mp4') for f in files):
        print(f"Found videos in: {root}")
        break 

In [16]:
!pip install -q evaluate accelerate